#  Custos de Importação

##  Objetivo

Transformar o arquivo `custos_importacao.json`, que possui dados aninhados, em um formato tabular adequado para análise e integração com banco de dados.

A estrutura final deve conter:

- `product_id` (integer)  
- `product_name` (text)  
- `category` (text)  
- `start_date` (date)  
- `usd_price` (float)  


O JSON possui um campo aninhado (`historic_data`) com múltiplos registros de preço por produto ao longo do tempo.

Abordagem adotada:

- Percorrer cada produto  
- Explodir o campo `historic_data`  
- Gerar uma linha para cada registro histórico  
- Consolidar os dados em formato tabular  

In [1]:
# Leitura do arquivo JSON


import json
import pandas as pd

with open("custos_importacao.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print(f"Quantidade de produtos no JSON: {len(data)}")

Quantidade de produtos no JSON: 150


## Normalização dos dados (Flatten)

Cada item dentro de `historic_data` será transformado em uma linha da tabela final.

In [2]:
#  Normalização dos dados


normalized_records = []

for product in data:
    product_id = int(product["product_id"])
    product_name = product["product_name"]
    category = product["category"]

    for history in product["historic_data"]:
        normalized_records.append({
            "product_id": product_id,
            "product_name": product_name,
            "category": category,
            "start_date": pd.to_datetime(
                history["start_date"],
                dayfirst=True
            ).date(),
            "usd_price": float(history["usd_price"])
        })

## Construção do DataFrame

In [3]:
df_importacao = pd.DataFrame(normalized_records)

df_importacao.head()

,product_id,product_name,category,start_date,usd_price
0,1,Transponder AIS Maré Magnum,eletrônicos,2016-08-10,10583.63
1,1,Transponder AIS Maré Magnum,eletrônicos,2018-06-15,8778.36
2,1,Transponder AIS Maré Magnum,eletrônicos,2018-09-25,8023.87
3,1,Transponder AIS Maré Magnum,eletrônicos,2019-03-19,8772.78
4,1,Transponder AIS Maré Magnum,eletrônicos,2020-01-17,7918.18


## Exportação para CSV

In [4]:
df_importacao.to_csv(
    "custos_importacao_normalizado.csv",
    index=False,
    encoding="utf-8"
)

## Validação da normalização

A quantidade total de registros no CSV deve ser igual à soma de todos os registros dentro de `historic_data`.

In [5]:
# Contagem final no DataFrame
total_registros = len(df_importacao)

print("Total de registros após normalização:", total_registros)

Total de registros após normalização: 1260


## Validação cruzada (direto no JSON)

In [8]:
len(df_importacao)

1260

In [7]:
total_registros = 0

for product in data:
    qtd_historicos = len(product["historic_data"])
    total_registros += qtd_historicos

print("Total de registros no JSON (antes da normalização):", total_registros)

Total de registros no JSON (antes da normalização): 1260


## Resultado

O processo de normalização gerou um total de:

👉 **1260 registros**